![image.png](https://i.imgur.com/a3uAqnb.png)



------------------------------<br>
Part 1: Why Transformers?<br>
------------------------------

Vaswani et al. (2017) introduced Transformers with the key idea: **attention is all you need** — no recurrence required.

In [ ]:
def task1_motivation():
    """
    Answer the following (write as comments):

    a) Name two limitations of RNNs/LSTMs that motivated Transformers.
    b) What are three key features of the Transformer architecture?
    c) Why is parallelization easier in Transformers than in RNNs?
    d) Name one task where the full encoder-decoder Transformer is commonly used.
    """
    # Your answer here:

    # --- Solution ---
    # a) Sequential computation (slow training) and vanishing gradients (poor long-range dependencies).
    # b) No recurrence (all attention), encoder-decoder structure, parallel processing, scales with data/compute.
    # c) All positions can be processed simultaneously; RNNs must compute h_t before h_{t+1}.
    # d) Machine translation, summarization, or text-to-text tasks (e.g., T5).

In [ ]:
def task1_big_picture():
    """
    Answer the following (write as comments):

    a) List some key engineering changes from 2017 to 2026 (from the lecture).
    b) In 2017, what positional encoding was used? Why is it problematic at scale?
    c) What is the attention complexity with sequence length N in standard MHA?
    """
    # Your answer here:

---

## Part 2: Self-Attention (25 min)

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

In **self-attention**, Q, K, and V all come from the same sequence — each token attends to all tokens.

------------------------------<br>
Quiz 2: Self-Attention — Theory<br>
------------------------------

In [ ]:
def task2_self_attention_theory():
    """
    Answer the following:

    a) In self-attention, where do Q, K, and V come from?
    b) What does the output of self-attention represent for each token?
    c) Why divide by sqrt(d_k) in the attention formula?
    """
    # Your answer here:

    # --- Solution ---
    # a) All are linear projections of the same input sequence X (Q=XW_Q, K=XW_K, V=XW_V).
    # b) A contextualized representation — a weighted blend of all token values based on relevance.
    # c) Prevents dot products from growing large in high dimensions, keeping softmax gradients stable.

------------------------------<br>
Part 3: Implement Self-Attention (15 min)<br>
------------------------------

Implement scaled dot-product self-attention for a small sequence. Use the input matrix `X` as Q, K, and V (no learned projections).

In [ ]:
def self_attention(X):
    """
    X: (seq_len, d_model)
    Returns: (output, attention_weights)
      output shape: (seq_len, d_model)
      attention_weights shape: (seq_len, seq_len)
    """
    # Your code here:

    # --- Solution ---
    d_k = X.shape[-1]
    scores = X @ X.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    output = weights @ X
    return output, weights


def task3_self_attention():
    # 3-token sequence, d_model=4
    X = np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.5, 0.5, 0.0, 0.0],
    ])
    output, weights = self_attention(X)
    # Each row of weights sums to 1; output blends token representations
    return output, weights

---

## Part 3: Multi-Head Attention

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\, W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

------------------------------<br>
Quiz 3: Multi-Head Attention<br>
------------------------------

In [ ]:
def task4_multihead_theory():
    """
    Answer the following:

    a) Why use multiple attention heads instead of one large head?
    b) What happens to the outputs of individual heads before the final output?
    c) Name one linguistic pattern different heads might capture.
    """
    # Your answer here:

    # --- Solution ---
    # a) Each head attends in a different representation subspace, capturing diverse relationships in parallel.
    # b) They are concatenated, then projected through W^O to the model dimension.
    # c) Syntax (subject-verb), coreference, long-distance dependencies, local vs global context, etc.

In [ ]:
## TASK: Implement Multi-Head Attention (10 min)

# Implement multi-head attention using the input matrix `X` as Q, K, and V (no learned projections).
# def multi_head_attention(X):
#     """
#     X: (seq_len, d_model)
#     Returns: (output, attention_weights)


---

## Part 4: Positional Encoding

Transformers have no inherent order — positional encodings are **added** to input embeddings:

$$PE(pos, 2i) = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

------------------------------<br>
Quiz 4: Positional Encoding — Theory <br>
------------------------------

In [ ]:
def task4_positional_theory():
    """
    Answer the following:

    a) Why do Transformers need positional encodings?
    b) Name one alternative to sinusoidal positional encoding.
    c) True or False: Positions close together in the sequence receive similar encoding perturbations.
    """
    # Your answer here:

    # --- Solution ---
    # a) Self-attention is permutation-invariant; without position info, word order is lost.
    # c) Learnable position embeddings, RoPE.
    # d) True — nearby positions get similar (though not identical) sinusoidal patterns.

------------------------------<br>
Exercise 4: Implement Sinusoidal Positional Encoding<br>
------------------------------

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """
    Return PE matrix of shape (max_len, d_model) using the sinusoidal formula.
    Even indices: sin, odd indices: cos.
    """
    # Your code here:


def task6_positional_encoding():
    pe = sinusoidal_positional_encoding(max_len=5, d_model=8)
    # pe[0] and pe[1] should be similar but not identical; all rows unique
    return pe

---

------------------------------<br>
Part 5: Architecture & Variants — Theory<br>
------------------------------

In [ ]:
def task6_architecture_theory():
    """
    Answer the following:

    a) What are the three main Transformer architecture variants? Give one example model each.
    b) What extra attention layer does the decoder have in encoder-decoder models?
    c) Compare Pre-LN vs Post-LN: which is more stable for training deep models?
    d) How does RMSNorm differ from LayerNorm?
    e) What is BERT's pre-training task? What is GPT's?
    f) How does RoPE differ from absolute sinusoidal encodings?
    g) Which 2026 models use RoPE? (Name two from the lecture.)
    """
    # Your answer here:


------------------------------<br>
Part 5 Exercise: Implement LayerNorm vs RMSNorm<br>
------------------------------

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-6):
    """x: (d,), gamma/beta: (d,)"""
    # Your code here:



def rms_norm(x, gamma, eps=1e-6):
    """x: (d,), gamma: (d,) — no bias term."""
    # Your code here:


def task5_norm_compare():
    x = np.array([1.0, -2.0, 3.0, -0.5])
    gamma = np.ones(4)
    beta = np.zeros(4)
    ln_out = layer_norm(x, gamma, beta)
    rms_out = rms_norm(x, gamma)
    return ln_out, rms_out